In [ ]:
!pip install -q openai opentelemetry-sdk

In [2]:
import json
import os
import sqlite3
import tempfile
import threading
import time
import urllib.request
import uuid
import warnings
from collections.abc import Callable, Iterator
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from contextvars import ContextVar
from dataclasses import dataclass, field
from pathlib import Path
from statistics import mean, median
from types import SimpleNamespace
from unittest.mock import patch

import numpy as np
from openai import OpenAI
from openai.types.chat import ChatCompletion
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

warnings.filterwarnings("ignore")

os.environ["OPENAI_API_KEY"] = ""

client = OpenAI()
MODEL = "gpt-5-mini"
FAST_MODEL = "gpt-5-nano"
EFFORT = "low"
MAX_STEPS = 8

# 1) Agent Operation

- An agent that works in a notebook is not yet a service.
- In production the questions change: why did this request take 40 seconds, why did yesterday cost three times more, which tool call failed, did last week's prompt change break anything?
- A single request is a tree of model calls and tool calls, each with its own latency, tokens and failure modes, and none of it is visible unless the harness records it.

In [3]:
PRICES = {
    "gpt-5-mini": {"input": 0.25, "cached": 0.025, "output": 2.00},
    "gpt-5-nano": {"input": 0.05, "cached": 0.005, "output": 0.40},
}
DB_URL = (
    "https://github.com/lerocha/chinook-database/raw/master/"
    "ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"
)
DB_PATH = Path(tempfile.gettempdir()) / "chinook.sqlite"
# Download Chinook DB to temp if missing
if not DB_PATH.exists():
    urllib.request.urlretrieve(DB_URL, DB_PATH)
# Open SQLite connection and create a lock
db = sqlite3.connect(DB_PATH, check_same_thread=False)
DB_LOCK = threading.Lock()
# Read database schema SQL for later use
SCHEMA = "\n\n".join(
    row[0]
    for row in db.execute(
        "SELECT sql FROM sqlite_master WHERE type='table' AND sql IS NOT NULL"
    )
)
print(f"{DB_PATH.stat().st_size / 1e6:.1f} MB, {SCHEMA.count('CREATE TABLE')} tables")

1.0 MB, 11 tables


In [4]:
QUESTIONS = {
    "How many customers are there?": "SELECT COUNT(*) FROM Customer",
    "How many tracks belong to the Rock genre?": (
        "SELECT COUNT(*) FROM Track t JOIN Genre g USING(GenreId) "
        "WHERE g.Name = 'Rock'"
    ),
    "Which country has the most customers?": (
        "SELECT Country, COUNT(*) c FROM Customer GROUP BY Country "
        "ORDER BY c DESC LIMIT 1"
    ),
    "Which artist has the most albums?": (
        "SELECT ar.Name, COUNT(*) c FROM Album al JOIN Artist ar USING(ArtistId) "
        "GROUP BY ar.ArtistId ORDER BY c DESC LIMIT 1"
    ),
    "Which sales support agent (full name) is responsible for the highest total "
    "invoice amount from their customers?": (
        "SELECT e.FirstName || ' ' || e.LastName, ROUND(SUM(i.Total), 2) s "
        "FROM Employee e JOIN Customer c ON c.SupportRepId = e.EmployeeId "
        "JOIN Invoice i ON i.CustomerId = c.CustomerId "
        "GROUP BY e.EmployeeId ORDER BY s DESC LIMIT 1"
    ),
    "Which genre generated the most revenue from invoice lines (unit price times "
    "quantity)?": (
        "SELECT g.Name, ROUND(SUM(il.UnitPrice * il.Quantity), 2) r "
        "FROM InvoiceLine il JOIN Track t USING(TrackId) JOIN Genre g USING(GenreId) "
        "GROUP BY g.GenreId ORDER BY r DESC LIMIT 1"
    ),
    "Which customer (full name) spent the most on Jazz tracks?": (
        "SELECT c.FirstName || ' ' || c.LastName, "
        "ROUND(SUM(il.UnitPrice * il.Quantity), 2) s FROM Customer c "
        "JOIN Invoice i USING(CustomerId) JOIN InvoiceLine il USING(InvoiceId) "
        "JOIN Track t USING(TrackId) JOIN Genre g USING(GenreId) "
        "WHERE g.Name = 'Jazz' GROUP BY c.CustomerId ORDER BY s DESC LIMIT 1"
    ),
    "What is the average number of tracks per album, rounded to two decimals?": (
        "SELECT ROUND(AVG(n), 2) FROM (SELECT COUNT(*) n FROM Track GROUP BY AlbumId)"
    ),
    "Which media type has the longest average track length in minutes, and what "
    "is that average (two decimals)?": (
        "SELECT m.Name, ROUND(AVG(t.Milliseconds) / 60000.0, 2) a FROM Track t "
        "JOIN MediaType m USING(MediaTypeId) GROUP BY m.MediaTypeId "
        "ORDER BY a DESC LIMIT 1"
    ),
    "How many customers have never bought a track from the Classical genre?": (
        "SELECT COUNT(*) FROM Customer WHERE CustomerId NOT IN (SELECT i.CustomerId "
        "FROM Invoice i JOIN InvoiceLine il USING(InvoiceId) JOIN Track t "
        "USING(TrackId) JOIN Genre g USING(GenreId) WHERE g.Name = 'Classical')"
    ),
}
# Run each question's SQL to build reference answers
REFERENCE = {q: db.execute(sql).fetchall() for q, sql in QUESTIONS.items()}
for question, rows in list(REFERENCE.items())[:3]:
    print(f"{question:<45} {rows}")

How many customers are there?                 [(59,)]
How many tracks belong to the Rock genre?     [(1297,)]
Which country has the most customers?         [('USA', 13)]


# 2) Implementation

## 2.1 Tracing

A **trace** is the tree of work for one request; each node is a **span** with a name, a start
and end time, a parent, and attributes. The OpenTelemetry GenAI semantic conventions fix the
attribute names, so any tracing backend can read them:

| Attribute | Meaning |
|---|---|
| `gen_ai.operation.name` | `invoke_agent`, `chat`, `execute_tool` |
| `gen_ai.request.model` | Model id sent in the request |
| `gen_ai.usage.input_tokens`, `gen_ai.usage.output_tokens` | Token counts from the response |
| `gen_ai.tool.name` | Tool being executed |

Attributes outside the convention (cached tokens, cost) go under an `app.` prefix. The current
span is held in a `ContextVar`, so nested `with` blocks become parent and child without passing
ids around.

In [5]:
@dataclass
class Span:
    """One timed unit of work in a trace."""

    name: str
    trace_id: str
    span_id: str
    parent_id: str | None
    start: float
    end: float = 0.0
    attributes: dict = field(default_factory=dict)
    status: str = "ok"

    @property
    def ms(self) -> float:
        # Demo tracer: create nested spans and print the trace
        """Duration in milliseconds."""
        return 1000 * (self.end - self.start)


CURRENT_SPAN: ContextVar[Span | None] = ContextVar("current_span", default=None)


class Tracer:
    """Collects finished spans; nesting comes from the current-span context variable."""

    def __init__(self) -> None:
        """Start with no spans."""
        self.spans: list[Span] = []
        self.lock = threading.Lock()

    @contextmanager
    def span(self, name: str, **attributes) -> Iterator[Span]:
        """Open a child of the current span (or a new trace) for the with-block."""
        parent = CURRENT_SPAN.get()
        span = Span(
            name=name,
            trace_id=parent.trace_id if parent else uuid.uuid4().hex[:16],
            span_id=uuid.uuid4().hex[:8],
            parent_id=parent.span_id if parent else None,
            start=time.perf_counter(),
            attributes=dict(attributes),
        )
        token = CURRENT_SPAN.set(span)
        try:
            yield span
        except Exception as error:
            span.status = f"error: {type(error).__name__}"
            raise
        finally:
            span.end = time.perf_counter()
            CURRENT_SPAN.reset(token)
            with self.lock:
                self.spans.append(span)

    def trace(self, trace_id: str) -> list[Span]:
        """All spans of one trace, in start order."""
        return sorted(
            (s for s in self.spans if s.trace_id == trace_id), key=lambda s: s.start
        )

In [6]:
def render(spans: list[Span], keys: tuple[str, ...] = ()) -> str:
    """Indented text view of a trace, showing selected attributes."""
    depth = {}
    lines = []
    for span in spans:
        depth[span.span_id] = depth.get(span.parent_id, -1) + 1
        shown = {
            k.split(".")[-1]: span.attributes[k] for k in keys if k in span.attributes
        }
        lines.append(
            f"{'  ' * depth[span.span_id]}{span.name:<24}{span.ms:>8.0f} ms  {shown}"
        )
    return "\n".join(lines)


# Create demo spans and render the trace output
demo_tracer = Tracer()
with demo_tracer.span("invoke_agent") as root:
    with demo_tracer.span("chat", **{"gen_ai.request.model": MODEL}):
        time.sleep(0.02)
    with demo_tracer.span("execute_tool", **{"gen_ai.tool.name": "run_sql"}):
        time.sleep(0.01)
print(
    render(
        demo_tracer.trace(root.trace_id), ("gen_ai.request.model", "gen_ai.tool.name")
    )
)

invoke_agent                  38 ms  {}
  chat                          25 ms  {'model': 'gpt-5-mini'}
  execute_tool                  13 ms  {'name': 'run_sql'}


The OpenTelemetry SDK implements the same model; in production its exporter sends spans to a
backend (Cloud Trace, Jaeger, Langfuse, Datadog) instead of a list. The equivalent of the cell
above, once, for reference:

In [7]:
# Configure OpenTelemetry in-memory exporter and tracer
exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))
otel_tracer = provider.get_tracer("agent-operation")
# Create spans and set an attribute on the chat span
with otel_tracer.start_as_current_span("invoke_agent"):
    with otel_tracer.start_as_current_span("chat") as chat_span:
        chat_span.set_attribute("gen_ai.request.model", MODEL)
for finished in exporter.get_finished_spans():
    parent = finished.parent.span_id if finished.parent else None
    print(f"{finished.name:<14} parent={parent} attributes={dict(finished.attributes)}")

chat           parent=6277463729609158736 attributes={'gen_ai.request.model': 'gpt-5-mini'}
invoke_agent   parent=None attributes={}


The agent is instrumented at two points: every model call becomes a `chat` span carrying model,
token counts and cost, and every tool call an `execute_tool` span. The agent has one tool,
`run_sql`, which returns at most 20 rows or the database error text, so the model can repair a
failed query.

In [8]:
# Create tracer and define the system prompt for the agent
tracer = Tracer()
SYSTEM_PROMPT = (
    "You are a SQL analyst for a music store. Answer the user's question by querying "
    "the SQLite database with the run_sql tool, then reply with a short answer that "
    "states the result. Database schema:\n\n" + SCHEMA
)
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "run_sql",
            "description": "Run one read-only SQLite query; returns up to 20 rows.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    }
]


def run_sql(query: str) -> str:
    """Execute a query and return rows as JSON, or the error text."""
    with tracer.span("execute_tool", **{"gen_ai.tool.name": "run_sql"}) as span:
        try:
            with DB_LOCK:
                rows = db.execute(query).fetchmany(20)
        except sqlite3.Error as error:
            span.status = "error: sqlite"
            return f"SQL error: {error}"
        return json.dumps(rows)


print(run_sql("SELECT Name FROM Genre LIMIT 3"))
print(run_sql("SELECT Nme FROM Genre"))

[["Rock"], ["Jazz"], ["Metal"]]
SQL error: no such column: Nme


In [9]:
def cost_usd(
    model: str, input_tokens: int, cached_tokens: int, output_tokens: int
) -> float:
    """List-price cost of one call."""
    price = PRICES[model]
    uncached = input_tokens - cached_tokens
    total = uncached * price["input"] + cached_tokens * price["cached"]
    return (total + output_tokens * price["output"]) / 1e6


def llm_call(
    model: str, messages: list[dict], tools: list[dict] | None = None
) -> ChatCompletion:
    """One traced chat completion with retries; records tokens and cost on the span."""
    attributes = {"gen_ai.operation.name": "chat", "gen_ai.request.model": model}
    with tracer.span("chat", **attributes) as span:
        for attempt in range(4):
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    reasoning_effort=EFFORT,
                    **({"tools": tools} if tools else {}),
                )
                break
            except Exception:
                if attempt == 3:
                    raise
                time.sleep(2**attempt)
        usage = response.usage
        details = usage.prompt_tokens_details
        cached = details.cached_tokens if details else 0
        span.attributes.update(
            {
                "gen_ai.usage.input_tokens": usage.prompt_tokens,
                "gen_ai.usage.output_tokens": usage.completion_tokens,
                "app.cached_tokens": cached,
                "app.cost_usd": cost_usd(
                    model, usage.prompt_tokens, cached, usage.completion_tokens
                ),
            }
        )
    return response

In [10]:
class BudgetExceeded(Exception):
    """Raised before a model call that the session can no longer afford."""


def run_agent(
    question: str,
    model: str = MODEL,
    budget: "Budget | None" = None,
    system_prompt: str = SYSTEM_PROMPT,
) -> tuple[str, str]:
    """Answer one question with the SQL tool; returns (answer, trace_id)."""
    attributes = {"gen_ai.operation.name": "invoke_agent", "app.model": model}
    with tracer.span("invoke_agent", **attributes) as root:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ]
        answer = "(no answer: step limit reached)"
        for _ in range(MAX_STEPS):
            if budget is not None:
                budget.check()
            response = llm_call(model, messages, TOOLS)
            if budget is not None:
                budget.charge(response)
            message = response.choices[0].message
            messages.append(message.model_dump(exclude_none=True))
            if not message.tool_calls:
                answer = message.content or ""
                break
            for tool_call in message.tool_calls:
                query = json.loads(tool_call.function.arguments).get("query", "")
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": run_sql(query),
                    }
                )
        root.attributes["app.answer"] = answer
    return answer, root.trace_id


# Run the agent on one question and print the trace output
answer, trace_id = run_agent(list(QUESTIONS)[4])
print(answer, "\n")
print(render(tracer.trace(trace_id), ("gen_ai.usage.output_tokens", "app.cost_usd")))

Jane Peacock is responsible for the highest total invoice amount, with total sales of 833.04. 

invoke_agent                3547 ms  {}
  chat                        2503 ms  {'output_tokens': 90, 'cost_usd': 0.00052625}
  execute_tool                   0 ms  {}
  chat                        1044 ms  {'output_tokens': 23, 'cost_usd': 0.0001305}


## 2.2 Metrics from traces

Metrics are aggregates over spans: no separate counters to keep in sync. A judge model checks
each answer against the reference query's result, so accuracy sits next to cost and latency.

Expect occasional high-latency outliers that require extra model calls and dominate cost; use p95 latency and individual traces to find and diagnose them.

In [11]:
JUDGE_PROMPT = """Question: {question}
Reference result (rows from the correct SQL query): {reference}
Agent answer: {answer}
Does the agent answer state the same result as the reference (allow rounding and
formatting differences)? Reply in JSON: {{"correct": true or false}}"""


def judge(question: str, answer: str) -> bool:
    """Judge-model check of one answer against the reference rows."""
    content = JUDGE_PROMPT.format(
        question=question, reference=REFERENCE[question], answer=answer
    )
    response = client.chat.completions.create(
        model=MODEL,
        reasoning_effort="minimal",
        messages=[{"role": "user", "content": content}],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)["correct"]


def trace_metrics(trace_id: str) -> dict:
    """Latency, call counts, tokens and cost for one request."""
    spans = tracer.trace(trace_id)
    chats = [s for s in spans if s.name == "chat"]
    root = next(s for s in spans if s.parent_id is None)
    return {
        "latency_s": root.ms / 1000,
        "llm_calls": len(chats),
        "tool_calls": sum(s.name == "execute_tool" for s in spans),
        "input_tokens": sum(s.attributes["gen_ai.usage.input_tokens"] for s in chats),
        "cached_tokens": sum(s.attributes["app.cached_tokens"] for s in chats),
        "output_tokens": sum(s.attributes["gen_ai.usage.output_tokens"] for s in chats),
        "cost_usd": sum(s.attributes["app.cost_usd"] for s in chats),
    }


print(trace_metrics(trace_id))

{'latency_s': 3.547321749967523, 'llm_calls': 2, 'tool_calls': 1, 'input_tokens': 2875, 'cached_tokens': 1280, 'output_tokens': 113, 'cost_usd': 0.00065675}


In [12]:
def evaluate(answer_fn: Callable[[str], tuple[str, str]]) -> list[dict]:
    """Run every question through answer_fn; per-question metrics and correctness."""

    def one(question: str) -> dict:
        """Answer, measure and judge one question."""
        answer, trace = answer_fn(question)
        return {
            "question": question,
            "correct": judge(question, answer),
            **trace_metrics(trace),
        }

    with ThreadPoolExecutor(5) as pool:
        return list(pool.map(one, QUESTIONS))


def summarize(rows: list[dict]) -> dict:
    """Accuracy, latency percentiles, mean tokens and cost per question."""
    latencies = [row["latency_s"] for row in rows]
    return {
        "accuracy": mean(row["correct"] for row in rows),
        "p50_s": median(latencies),
        "p95_s": float(np.percentile(latencies, 95)),
        "output_tokens": mean(row["output_tokens"] for row in rows),
        "cost_per_q_usd": mean(row["cost_usd"] for row in rows),
    }


# Evaluate baseline model on all questions and print rows
baseline = evaluate(lambda question: run_agent(question, MODEL))
print(f"{'question':<48}{'ok':>4}{'calls':>6}{'s':>6}{'out tok':>9}{'$':>9}")
for row in baseline:
    print(
        f"{row['question'][:46]:<48}{row['correct']!s:>4}{row['llm_calls']:>6}"
        f"{row['latency_s']:>6.1f}{row['output_tokens']:>9}{row['cost_usd']:>9.5f}"
    )
print({key: round(value, 5) for key, value in summarize(baseline).items()})

question                                          ok calls     s  out tok        $
How many customers are there?                   True     2   2.2       40  0.00049
How many tracks belong to the Rock genre?       True     2   2.7       64  0.00028
Which country has the most customers?           True     2   2.6       59  0.00027
Which artist has the most albums?               True     2   2.9       83  0.00033
Which sales support agent (full name) is respo  True     2   3.5      113  0.00040
Which genre generated the most revenue from in  True     2   4.0      165  0.00076
Which customer (full name) spent the most on J  True     2   3.9      148  0.00050
What is the average number of tracks per album  True     2   4.2      166  0.00050
Which media type has the longest average track  True     2   3.2      107  0.00041
How many customers have never bought a track f  True     2   3.6      125  0.00042
{'accuracy': 1, 'p50_s': 3.33547, 'p95_s': 4.11967, 'output_tokens': 107, 'cost_per_q_u

## 2.3 Token and spend budgets

A budget is checked **before** each model call, because the bill arrives after. It caps tokens
and dollars per session; when exceeded, the request ends with an explicit error the caller can
handle (return a partial answer, ask the user, escalate) instead of looping on.

Expect budget checks to stop sessions that would exceed configured token or spend limits; in production, set budgets above typical p95 costs so they only trigger on runaway sessions.

In [13]:
@dataclass
class Budget:
    """Per-session caps on tokens and spend, checked before each call."""

    max_tokens: int
    max_usd: float
    model: str = MODEL
    tokens: int = 0
    usd: float = 0.0

    def check(self) -> None:
        """Raise if the session has already used its budget."""
        if self.tokens >= self.max_tokens or self.usd >= self.max_usd:
            raise BudgetExceeded(f"used {self.tokens} tokens, ${self.usd:.5f}")

    def charge(self, response: ChatCompletion) -> None:
        """Add one response's usage to the session totals."""
        usage = response.usage
        details = usage.prompt_tokens_details
        cached = details.cached_tokens if details else 0
        self.tokens += usage.prompt_tokens + usage.completion_tokens
        self.usd += cost_usd(
            self.model, usage.prompt_tokens, cached, usage.completion_tokens
        )


# Demonstrate budget enforcement by running an agent
tight = Budget(max_tokens=1_000, max_usd=0.01)
try:
    answer, _ = run_agent(list(QUESTIONS)[6], budget=tight)
    print("finished within budget:", answer)
except BudgetExceeded as error:
    print("stopped before the next call:", error)

stopped before the next call: used 1510 tokens, $0.00061


## 2.4 Prompt caching

The provider caches the longest previously seen **prefix** of a prompt (from 1,024 tokens, in
128-token steps) and bills those tokens at the cached rate. The schema-heavy system prompt is
the stable prefix here. One common mistake is to put something that changes per request at the
start, such as the current time; this experiment measures the cached share both ways.

Expect caching to reduce billed prompt tokens when prefixes are stable; changing early tokens prevents cache hits.

In [14]:
def cache_experiment(prefix_varies: bool, num_calls: int = 5) -> list[int]:
    """Cached tokens per call for repeated requests with a stable or varying prefix."""
    cached = []
    for call_index in range(num_calls):
        stamp = f"Request time: {time.time():.6f}\n" if prefix_varies else ""
        messages = [
            {"role": "system", "content": stamp + SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Name one table in the schema. ({call_index})",
            },
        ]
        response = llm_call(MODEL, messages)
        details = response.usage.prompt_tokens_details
        cached.append(details.cached_tokens if details else 0)
        time.sleep(2)
    return cached


# Run cache experiment with stable vs varying prefixes
stable = cache_experiment(prefix_varies=False)
varying = cache_experiment(prefix_varies=True)
prompt_tokens = tracer.spans[-1].attributes["gen_ai.usage.input_tokens"]
print(f"prompt tokens per call: {prompt_tokens}")
print(f"stable prefix  cached tokens: {stable}")
print(f"varying prefix cached tokens: {varying}")

prompt tokens per call: 1262
stable prefix  cached tokens: [0, 1152, 1152, 1152, 1152]
varying prefix cached tokens: [0, 0, 0, 0, 0]


Caching is
best-effort (requests can land on a server without the cache), so treat hit rate as a metric to
monitor, not a guarantee. The rule for prompt layout: stable content first (instructions, tool
definitions, schemas, few-shot examples), per-request content last.

## 2.5 Model routing

A router sends each request to the cheap or the strong model. The decision is made by a model
call (`gpt-5-nano`, minimal effort) that classifies the question, because "is this a simple
lookup or a multi-join analysis" is a judgement, not a keyword match.

Expect routing behavior to be workload-specific. Measure on your own question set before deploying a router to see the actual trade-offs.

In [15]:
ROUTER_PROMPT = """A SQL agent will answer this question about a music-store database.
Question: {question}
Is it simple (one table or one join, a count or a lookup) or complex (several
joins, aggregation over groups, subqueries)? Reply in JSON:
{{"route": "simple" or "complex"}}"""


def route(question: str) -> str:
    """Model id chosen by the router for one question."""
    with tracer.span("route", **{"gen_ai.request.model": FAST_MODEL}) as span:
        response = client.chat.completions.create(
            model=FAST_MODEL,
            reasoning_effort="minimal",
            messages=[
                {"role": "user", "content": ROUTER_PROMPT.format(question=question)}
            ],
            response_format={"type": "json_object"},
        )
        choice = json.loads(response.choices[0].message.content)["route"]
        usage = response.usage
        span.attributes["app.cost_usd"] = cost_usd(
            FAST_MODEL, usage.prompt_tokens, 0, usage.completion_tokens
        )
    return FAST_MODEL if choice == "simple" else MODEL


def routed_agent(question: str) -> tuple[str, str]:
    """Route, then run the agent under one trace that includes the router span."""
    with tracer.span("request") as root:
        run_agent(question, route(question))
    spans = tracer.trace(root.trace_id)
    answer = next(s.attributes["app.answer"] for s in spans if s.name == "invoke_agent")
    return answer, root.trace_id


# Show routing decisions for a few example questions
for question in list(QUESTIONS)[:2] + list(QUESTIONS)[-2:]:
    print(f"{route(question):<12}{question[:70]}")

gpt-5-nano  How many customers are there?


gpt-5-nano  How many tracks belong to the Rock genre?


gpt-5-mini  Which media type has the longest average track length in minutes, and 


gpt-5-mini  How many customers have never bought a track from the Classical genre?


In [16]:
def trace_cost(trace_id: str) -> float:
    """Total cost of a trace including router spans."""
    return sum(s.attributes.get("app.cost_usd", 0.0) for s in tracer.trace(trace_id))


def evaluate_routed() -> list[dict]:
    """Like evaluate, but cost includes the router call."""
    rows = []
    for question in QUESTIONS:
        answer, trace = routed_agent(question)
        metrics = trace_metrics(trace) | {"cost_usd": trace_cost(trace)}
        rows.append(
            {"question": question, "correct": judge(question, answer), **metrics}
        )
    return rows


# Compare routing configurations and print summaries
ROUTING = {
    f"all {MODEL}": summarize(baseline),
    f"all {FAST_MODEL}": summarize(evaluate(lambda q: run_agent(q, FAST_MODEL))),
    "routed": summarize(evaluate_routed()),
}
print(f"{'config':<18}{'accuracy':>9}{'p50 s':>7}{'p95 s':>7}{'$ per question':>16}")
for config, row in ROUTING.items():
    print(
        f"{config:<18}{row['accuracy']:>9.2f}{row['p50_s']:>7.1f}{row['p95_s']:>7.1f}"
        f"{row['cost_per_q_usd']:>16.5f}"
    )

config             accuracy  p50 s  p95 s  $ per question
all gpt-5-mini         1.00    3.3    4.1         0.00044
all gpt-5-nano         1.00    4.5    7.4         0.00014
routed                 1.00    3.8    5.3         0.00055


The three rows place the configurations on an accuracy–cost plane. If the cheap model alone is
already as accurate on this workload, routing only adds the router's cost and latency; if it
fails on the complex questions, routing keeps most of the saving while recovering accuracy. The
answer is workload-specific, which is the point: measure on your own question set before adding
a router.

## 2.6 Testing

Two kinds of tests, for two kinds of change:

| Test | Model | Catches | Runs |
|---|---|---|---|
| Unit test of the harness | Scripted fake: fixed responses, no API | Loop bugs: tool errors not returned, step limit not enforced, budget not checked | Every commit, in milliseconds |
| Golden regression set | The real model | Prompt, model or tool changes that make answers wrong | Before every release |

The scripted model replaces `llm_call` with a function that returns prepared responses in the
same shape as the API's.

Expect to use the golden set as a release gate that enforces minimum accuracy and bounds on cost and latency; test changes against a baseline.

In [17]:
def scripted_response(tool_query: str | None = None, text: str = "") -> SimpleNamespace:
    """A fake ChatCompletion: either one run_sql tool call or a final text answer."""
    tool_calls = None
    if tool_query is not None:
        function = SimpleNamespace(
            name="run_sql", arguments=json.dumps({"query": tool_query})
        )
        tool_calls = [SimpleNamespace(id=uuid.uuid4().hex[:8], function=function)]
    message = SimpleNamespace(content=text, tool_calls=tool_calls)
    message.model_dump = lambda exclude_none: {"role": "assistant", "content": text}
    usage = SimpleNamespace(
        prompt_tokens=1_000, completion_tokens=50, prompt_tokens_details=None
    )
    return SimpleNamespace(choices=[SimpleNamespace(message=message)], usage=usage)


def scripted_llm(responses: list[SimpleNamespace]) -> Callable:
    """llm_call replacement that returns the responses in order and records messages."""
    seen = []

    def fake(
        model: str, messages: list[dict], tools: list[dict] | None = None
    ) -> SimpleNamespace:
        """Return the next scripted response."""
        seen.append(list(messages))
        return responses[min(len(seen), len(responses)) - 1]

    fake.seen = seen
    return fake

In [18]:
def test_tool_error_reaches_model() -> None:
    """A failing query's error text is sent back so the model can repair it."""
    fake = scripted_llm(
        [scripted_response("SELECT Nme FROM Genre"), scripted_response(text="done")]
    )
    with patch(f"{__name__}.llm_call", fake):
        answer, _ = run_agent("any question")
    assert answer == "done"
    assert "SQL error" in fake.seen[1][-1]["content"]


def test_step_limit() -> None:
    """A model that never stops calling tools is cut off at MAX_STEPS."""
    fake = scripted_llm([scripted_response("SELECT 1")])
    with patch(f"{__name__}.llm_call", fake):
        answer, _ = run_agent("any question")
    assert len(fake.seen) == MAX_STEPS and answer.startswith("(no answer")


def test_budget_stops_session() -> None:
    """The budget check runs before the call that would exceed it."""
    fake = scripted_llm([scripted_response("SELECT 1")])
    with patch(f"{__name__}.llm_call", fake):
        try:
            run_agent("any question", budget=Budget(max_tokens=2_000, max_usd=1.0))
        except BudgetExceeded:
            pass
    assert len(fake.seen) == 2


# Run tests and print timing for each
for test in (test_tool_error_reaches_model, test_step_limit, test_budget_stops_session):
    start = time.perf_counter()
    test()
    print(f"pass  {test.__name__:<32}{1000 * (time.perf_counter() - start):.1f} ms")

pass  test_tool_error_reaches_model   16.0 ms
pass  test_step_limit                 0.1 ms
pass  test_budget_stops_session       0.1 ms


The golden set is the question list with its reference results, run against the real model.
A release gate states the minimum accuracy and the maximum cost and latency; a prompt or model
change ships only if it passes.

In [19]:
GATE = {"accuracy": 0.8, "p95_s": 60.0, "cost_per_q_usd": 0.01}


def release_gate(summary: dict) -> list[str]:
    """Names of the gate conditions the summary fails."""
    failures = []
    if summary["accuracy"] < GATE["accuracy"]:
        failures.append("accuracy")
    if summary["p95_s"] > GATE["p95_s"]:
        failures.append("p95 latency")
    if summary["cost_per_q_usd"] > GATE["cost_per_q_usd"]:
        failures.append("cost")
    return failures


# Check release gate conditions for each routing summary
for config, row in ROUTING.items():
    failures = release_gate(row)
    print(f"{config:<18}{'PASS' if not failures else 'FAIL: ' + ', '.join(failures)}")

all gpt-5-mini    PASS
all gpt-5-nano    PASS
routed            PASS


# 3) End-to-end request

A production request combines the pieces: the router picks the model, a budget guards the
session, the stable system prompt is cached, every step is traced, and the trace yields the
metrics a dashboard aggregates.

In [20]:
question = list(QUESTIONS)[6]
# Run one routed request with a session budget and print trace
with tracer.span("request", **{"app.user": "analyst-42"}) as root:
    model_id = route(question)
    session_budget = Budget(max_tokens=60_000, max_usd=0.05, model=model_id)
    answer, _ = run_agent(question, model_id, budget=session_budget)
print("answer:", answer, "\n")
keys = ("gen_ai.request.model", "gen_ai.usage.output_tokens", "app.cached_tokens")
print(render(tracer.trace(root.trace_id), keys))
request_cost = trace_cost(root.trace_id)
print(f"\ncost ${request_cost:.5f}, budget used {session_budget.tokens} tokens")

answer: Dominique Lefebvre spent the most on Jazz tracks, totaling $5.94. 

request                     4824 ms  {}
  route                        830 ms  {'model': 'gpt-5-nano'}
  invoke_agent                3993 ms  {}
    chat                        2790 ms  {'model': 'gpt-5-mini', 'output_tokens': 127, 'cached_tokens': 1152}
    execute_tool                   0 ms  {}
    chat                        1203 ms  {'model': 'gpt-5-mini', 'output_tokens': 21, 'cached_tokens': 1408}

cost $0.00046, budget used 3054 tokens


# Summary

| Concern | Practice |
|---|---|
| What to trace | Every model call, tool call and retrieval as a span; prompts and outputs as span events, sampled and with PII redacted |
| Where traces go | An OpenTelemetry exporter to the platform's backend (Cloud Trace, Datadog, Langfuse, LangSmith); keep `gen_ai.*` names so dashboards work across frameworks |
| What to alert on | p95 latency, error rate per tool, cost per request, cache hit rate, step count per request (loops) |
| Budgets | Per session and per tenant; set from measured p95 cost; fail with a clear message, not a timeout |
| Caching | Stable prefix first; monitor hit rate after every prompt change; provider caches expire after minutes of inactivity |
| Routing | Measure cheap-model accuracy per route on your own set; keep the router call minimal effort; log the route on the trace |
| Testing | Scripted-model unit tests on every commit; golden set with a release gate before every prompt or model change |
| Retries | Back off on rate-limit and 5xx errors with jitter; do not retry on 4xx; record retries on the span |

**Failure modes.** Judges drift when the judge model changes, so pin it. Golden sets go stale as
the data changes (the reference SQL must be re-run, not the stored answer reused). Price tables
change; read them from configuration, not code.

| Mechanism | Built here | Measured |
|---|---|---|
| Tracing | `Tracer` with context-variable nesting and GenAI attribute names | Span tree per request |
| Metrics | Aggregates over spans | Accuracy, p50/p95 latency, tokens, cost per question |
| Budgets | Pre-call token and spend check | Session stopped before overspending |
| Prompt caching | Stable vs varying prefix | Cached tokens per call |
| Routing | Router model + two agent models | Accuracy vs cost per configuration |
| Testing | Scripted-model unit tests; golden set and release gate | Harness behaviour; release pass/fail |